In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://iabd-spark-master:7077")
    .appName("spark4-full-stack-demo")
    # --- Hive Metastore en MySQL (config explicita) ---
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.warehouse.dir", "s3a://warehouse/hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:mysql://iabd-mysql:3306/hive_metastore?useSSL=false&allowPublicKeyRetrieval=true")
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "com.mysql.cj.jdbc.Driver")
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", "hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", "hivepass")
    .config("spark.hadoop.hive.metastore.schema.verification", "false")
    .config("spark.hadoop.hive.metastore.schema.verification.record.version", "false")
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "false")
    .config("spark.hadoop.datanucleus.autoCreateSchema", "false")
    .config("spark.hadoop.datanucleus.fixedDatastore", "true")
    .config("spark.hadoop.datanucleus.schema.autoCreateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateConstraints", "false")
    .config("spark.hadoop.datanucleus.schema.validateColumns", "false")
    # --- JDBC driver + extra jars (driver y executors) ---
    .config("spark.driver.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
     # --- MinIO / S3A ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://iabd-minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        
    .enableHiveSupport()
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 12:07:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.sql("show databases").show()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [3]:
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='s3a://warehouse/hive'),
 Database(name='demo', catalog='spark_catalog', description='', locationUri='s3a://warehouse/hive/demo.db')]

In [4]:
spark.catalog.currentDatabase()

'default'

In [5]:
spark.sql("create database if not exists s8a")

26/04/29 12:07:07 WARN ObjectStore: Failed to get database s8a, returning NoSuchObjectException
26/04/29 12:07:07 WARN ObjectStore: Failed to get database s8a, returning NoSuchObjectException
26/04/29 12:07:07 WARN ObjectStore: Failed to get database s8a, returning NoSuchObjectException


DataFrame[]

In [6]:
spark.sql("use s8a")

DataFrame[]

In [7]:
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='s3a://warehouse/hive'),
 Database(name='demo', catalog='spark_catalog', description='', locationUri='s3a://warehouse/hive/demo.db'),
 Database(name='s8a', catalog='spark_catalog', description='', locationUri='s3a://warehouse/hive/s8a.db')]

In [8]:
jdbcDF = spark.read \
    .format("jdbc") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("url", "jdbc:mysql://iabd-mysql") \
    .option("dbtable", "retail_db.customers") \
    .option("port", "3306") \
    .option("user", "iabd") \
    .option("password", "iabd") \
    .load()
jdbcDF.createOrReplaceTempView("clientes")

In [9]:
spark.catalog.listTables()

[Table(name='clientes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [10]:
jdbcDF.count()

12435